In [3]:
import urllib.request
from pathlib import Path

MODEL_PATH = 'holistic_landmarker.task'

if not Path(MODEL_PATH).exists():

    print('Downloading holistic_landmarker.task...')

    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/'
        'holistic_landmarker/holistic_landmarker/'
        'float16/latest/holistic_landmarker.task',
        MODEL_PATH
    )

    print('✓ Download complete')

else:
    print('✓ holistic_landmarker.task already exists')

✓ Download complete


In [4]:
# ─────────────────────────────────────────────────────────────
# MODERN MEDIAPIPE TASKS API VERSION
# Sign Language Recognition Training + ONNX Export
# NO mp.solutions usage
# ─────────────────────────────────────────────────────────────

import cv2
import torch
import random
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

from sklearn.model_selection import train_test_split

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import onnx
import onnxruntime as ort

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────

BASE = Path('isl_nmf')

DATA_DIR = BASE / 'data' / 'wlasl'

ONNX_DIR = BASE / 'onnx'
ONNX_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 30

BATCH_SIZE = 32

EPOCHS = 20

LR = 1e-3

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ─────────────────────────────────────────────────────────────
# MEDIAPIPE TASKS API
# ─────────────────────────────────────────────────────────────

MODEL_PATH = 'holistic_landmarker.task'

# Download manually:
# https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task

base_options = python.BaseOptions(
    model_asset_path=MODEL_PATH
)

options = vision.HolisticLandmarkerOptions(

    base_options=base_options,

    running_mode=vision.RunningMode.IMAGE,

    output_face_blendshapes=False,

    output_segmentation_mask=False
)

holistic = vision.HolisticLandmarker.create_from_options(options)

print('✓ MediaPipe Tasks API loaded')

# ─────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────

class_names = sorted([
    d.name for d in DATA_DIR.iterdir()
    if d.is_dir()
])

label2id = {
    c: i for i, c in enumerate(class_names)
}

id2label = {
    i: c for c, i in label2id.items()
}

samples = []

for cls in class_names:

    vids = list((DATA_DIR / cls).glob('*.mp4'))

    for v in vids:

        samples.append({
            'video': str(v),
            'label': label2id[cls]
        })

df = pd.DataFrame(samples)

print(f'\nVideos: {len(df)}')
print(f'Classes: {len(class_names)}')

# ─────────────────────────────────────────────────────────────
# LANDMARK EXTRACTION
# ─────────────────────────────────────────────────────────────

def extract_landmarks(result):

    feats = []

    # ─────────────────────────────────────────
    # LEFT HAND
    # ─────────────────────────────────────────

    if result.left_hand_landmarks:

        for lm in result.left_hand_landmarks[0]:

            feats.extend([
                lm.x,
                lm.y,
                lm.z
            ])

    else:
        feats.extend([0.0] * 63)

    # ─────────────────────────────────────────
    # RIGHT HAND
    # ─────────────────────────────────────────

    if result.right_hand_landmarks:

        for lm in result.right_hand_landmarks[0]:

            feats.extend([
                lm.x,
                lm.y,
                lm.z
            ])

    else:
        feats.extend([0.0] * 63)

    # ─────────────────────────────────────────
    # POSE
    # ─────────────────────────────────────────

    if result.pose_landmarks:

        pose_ids = [
            11, 12,
            13, 14,
            15, 16
        ]

        pose_landmarks = result.pose_landmarks[0]

        for idx in pose_ids:

            lm = pose_landmarks[idx]

            feats.extend([
                lm.x,
                lm.y,
                lm.z
            ])

    else:
        feats.extend([0.0] * 18)

    return np.array(feats, dtype=np.float32)

# ─────────────────────────────────────────────────────────────
# VIDEO PROCESSING
# ─────────────────────────────────────────────────────────────

def process_video(video_path):

    cap = cv2.VideoCapture(video_path)

    seq = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        result = holistic.detect(mp_image)

        feats = extract_landmarks(result)

        seq.append(feats)

    cap.release()

    if len(seq) == 0:
        return None

    seq = np.array(seq, dtype=np.float32)

    # ─────────────────────────────────────────
    # TEMPORAL NORMALIZATION
    # ─────────────────────────────────────────

    if len(seq) > SEQ_LEN:

        idxs = np.linspace(
            0,
            len(seq) - 1,
            SEQ_LEN
        ).astype(int)

        seq = seq[idxs]

    elif len(seq) < SEQ_LEN:

        pad = np.zeros(
            (SEQ_LEN - len(seq), seq.shape[1]),
            dtype=np.float32
        )

        seq = np.concatenate([seq, pad])

    return seq

# ─────────────────────────────────────────────────────────────
# BUILD FEATURES
# ─────────────────────────────────────────────────────────────

X = []
y = []

print('\nExtracting landmarks...\n')

for _, row in tqdm(df.iterrows(), total=len(df)):

    seq = process_video(row['video'])

    if seq is None:
        continue

    X.append(seq)

    y.append(row['label'])

X = np.array(X, dtype=np.float32)

y = np.array(y)

print(f'\nFeature shape: {X.shape}')

# ─────────────────────────────────────────────────────────────
# TRAIN / VAL
# ─────────────────────────────────────────────────────────────

X_tr, X_val, y_tr, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

# ─────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────

class SignDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.float32)

        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

train_ds = SignDataset(X_tr, y_tr)

val_ds = SignDataset(X_val, y_val)

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ─────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────

INPUT_DIM = X.shape[-1]

N_CLASSES = len(class_names)

class SignTCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Conv1d(INPUT_DIM, 128, 3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Conv1d(128, 256, 3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Conv1d(256, 256, 3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.fc = nn.Linear(256, N_CLASSES)

    def forward(self, x):

        x = x.permute(0, 2, 1)

        x = self.net(x)

        x = x.squeeze(-1)

        logits = self.fc(x)

        return logits

model = SignTCN().to(DEVICE)

# ─────────────────────────────────────────────────────────────
# TRAIN
# ─────────────────────────────────────────────────────────────

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

best_acc = 0

print('\nTraining...\n')

for epoch in range(EPOCHS):

    # TRAIN

    model.train()

    tr_correct = 0
    tr_total = 0

    for Xb, yb in train_dl:

        Xb = Xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        logits = model(Xb)

        loss = criterion(logits, yb)

        loss.backward()

        optimizer.step()

        preds = logits.argmax(1)

        tr_correct += (preds == yb).sum().item()

        tr_total += len(yb)

    train_acc = tr_correct / tr_total

    # VAL

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for Xb, yb in val_dl:

            Xb = Xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(Xb)

            preds = logits.argmax(1)

            val_correct += (preds == yb).sum().item()

            val_total += len(yb)

    val_acc = val_correct / val_total

    print(
        f'Epoch {epoch+1:02d}/{EPOCHS} | '
        f'Train={train_acc:.1%} | '
        f'Val={val_acc:.1%}'
    )

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            BASE / 'best_sign_model.pt'
        )

print(f'\nBest validation accuracy: {best_acc:.1%}')

# ─────────────────────────────────────────────────────────────
# LOAD BEST
# ─────────────────────────────────────────────────────────────

model.load_state_dict(
    torch.load(BASE / 'best_sign_model.pt')
)

model.eval()

# ─────────────────────────────────────────────────────────────
# EXPORT ONNX
# ─────────────────────────────────────────────────────────────

dummy = torch.randn(
    1,
    SEQ_LEN,
    INPUT_DIM
).to(DEVICE)

onnx_path = ONNX_DIR / 'sign_language_tcn.onnx'

torch.onnx.export(

    model,

    dummy,

    str(onnx_path),

    export_params=True,

    opset_version=18,

    do_constant_folding=True,

    input_names=['landmark_sequence'],

    output_names=['sign_logits'],

    dynamic_axes={

        'landmark_sequence': {
            0: 'B',
            1: 'T'
        },

        'sign_logits': {
            0: 'B'
        }
    }
)

print(f'\n✓ ONNX exported:')
print(onnx_path)

# ─────────────────────────────────────────────────────────────
# VERIFY
# ─────────────────────────────────────────────────────────────

onnx_model = onnx.load(str(onnx_path))

onnx.checker.check_model(onnx_model)

print('✓ ONNX verified')

# ─────────────────────────────────────────────────────────────
# TEST ONNX
# ─────────────────────────────────────────────────────────────

sess = ort.InferenceSession(
    str(onnx_path),
    providers=['CPUExecutionProvider']
)

sample = X_val[:1]

logits = sess.run(
    ['sign_logits'],
    {'landmark_sequence': sample.astype(np.float32)}
)[0]

pred = logits.argmax(1)[0]

print('\nPrediction:')
print(id2label[pred])

print('\nDONE.')

I0000 00:00:1778491573.164573 1856772 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778491573.228755 1856774 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778491573.240141 1856778 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778491573.241952 1856778 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778491573.242095 1856777 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778491573.242328 1856776 inference_feedback_manager.cc:121] Feedback manager requires a mod

✓ MediaPipe Tasks API loaded


FileNotFoundError: [Errno 2] No such file or directory: 'isl_nmf/data/wlasl'